# Image Classification with Hotswappable Architectures

This notebook trains image classifiers with **configurable architectures** (CNN or MLP) on various datasets, then evaluates performance with metrics and visualizations.

**Supported architectures:**
- **CNN**: Convolutional Neural Network with multiple conv blocks
- **MLP**: Multi-Layer Perceptron (fully connected layers)

**Supported datasets:**
- **CIFAR-10**: 32x32 color images in 10 classes (airplanes, cars, birds, etc.)
- **MNIST**: 28x28 grayscale handwritten digits (0-9)
- **Fashion-MNIST**: 28x28 grayscale clothing items (t-shirts, trousers, shoes, etc.)

The model architecture automatically adapts to the input dimensions and number of channels for each dataset.

Now we'll import the necessary libraries and enable autoreload so changes to our shared library are automatically loaded.

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as L
import wandb
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np

# Import shared utilities from local package
from aiml_notebooks import (
    log_gradients, log_model_weights, log_gradient_flow,
    create_trainer, create_dataset, create_dataloaders,
    show_image_grid_normalized, plot_confusion_matrix,
    get_dataset_config,
    log_confusion_matrix_callback, log_prediction_grid_callback,
    create_image_classifier,  # Factory for CNN/MLP classifiers
)

# Enable autoreload for hot reloading of library changes
%load_ext autoreload
%autoreload 2

print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {L.__version__}")

In [ ]:
# Configuration (Base defaults - can be overridden by papermill parameters)
CONFIG = {
    # Seeds
    'seed': 42,                      # Random seed for reproducibility

    # Data
    'dataset': 'mnist',              # Dataset to use: 'cifar10', 'mnist', or 'fashionmnist'
    'batch_size': 128,               # Number of examples per training batch
    'num_workers': 4,                # Number of parallel data loading workers
    'persistent_workers': True,
    'pin_memory': True,
    
    # Training
    'max_epochs': 50,                # Number of complete passes through training data
    'log_every_n_steps': 20,         # How often to log training metrics
    
    # Model Architecture
    'classifier_type': 'mlp',        # Classifier architecture: 'cnn' or 'mlp'
    'dropout': 0.5,                  # Dropout rate to prevent overfitting
    'learning_rate': 1e-3,           # Step size for optimizer (0.001)
    
    # MLP-specific parameters (used when classifier_type='mlp')
    'mlp_hidden_sizes': [512, 256, 128], # List of hidden layer sizes for MLP
    
    # CNN-specific parameters (used when classifier_type='cnn')
    'cnn_num_conv_layers': 3,        # Number of convolutional blocks
    'cnn_base_channels': 32,         # Number of channels in first conv layer (doubles each block)
    
    # Weights & Biases
    'wandb_project': "classification-image",  # W&B project name (auto-detected from notebook filename)
    'wandb_run_name': None,          # Optional run name (None = auto-generated)
}

# Set random seeds
L.seed_everything(CONFIG['seed'])

Now we'll use the dataset factory to load the selected dataset:
- **MNIST**: 70,000 28x28 grayscale handwritten digits (60k train, 10k test)
- **Fashion-MNIST**: 70,000 28x28 grayscale clothing images in 10 classes (60k train, 10k test)
- **CIFAR-10**: 60,000 32x32 color images in 10 classes (50k train, 10k test)

In [ ]:
# Get dataset configuration using shared library function
dataset_config = get_dataset_config(CONFIG['dataset'])
class_names = dataset_config['classes']
num_classes = len(class_names)
num_channels = dataset_config['num_channels']
image_size = dataset_config['image_size']

# Define data transforms (normalization and data augmentation for training)
# Training transforms with data augmentation
if CONFIG['dataset'] == 'cifar10':
    # CIFAR-10 specific augmentation
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(image_size, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(dataset_config['mean'], dataset_config['std'])
    ])
elif CONFIG['dataset'] in ['mnist', 'fashionmnist']:
    # MNIST/Fashion-MNIST specific augmentation
    train_transform = transforms.Compose([
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(dataset_config['mean'], dataset_config['std'])
    ])

# Test transforms (no augmentation)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(dataset_config['mean'], dataset_config['std'])
])

# Load dataset using the factory (returns train, test)
train_dataset, test_dataset = create_dataset(
    dataset_id=CONFIG['dataset'],
    train_transform=train_transform,
    test_transform=test_transform
)

print(f"Dataset: {dataset_config['name']}")
print(f"Image size: {image_size}x{image_size}x{num_channels}")
print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Classes ({num_classes}): {', '.join(class_names)}")

Now we'll use the dataloader factory to create batched, shuffled loaders for training and testing.

The model will automatically log visualizations (confusion matrices and prediction grids) to W&B at the end of each validation epoch.

In [ ]:
# Create data loaders using the factory
train_loader, test_loader = create_dataloaders(
    train_dataset=train_dataset,
    val_dataset=test_dataset,  # Using test set as val set for this notebook
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
    pin_memory=CONFIG['pin_memory'],
    persistent_workers=CONFIG['persistent_workers'],
    use_collate_fn=False,  # Vision datasets don't need padding
)

print(f"✓ Data loaders created (batch_size={CONFIG['batch_size']})")

Now we'll create our classifier using the configuration. The architecture is **hotswappable** - simply change `classifier_type` in the config to switch between CNN and MLP.

**CNN Architecture:**
- Multiple convolutional blocks (Conv → BatchNorm → ReLU → Conv → BatchNorm → ReLU → MaxPool)
- Fully connected classifier with dropout

**MLP Architecture:**
- Flatten input image
- Multiple fully connected layers with BatchNorm, ReLU, and dropout
- Output layer for classification

In [ ]:
# Create the classifier using the factory function
# This automatically selects CNN or MLP based on CONFIG['classifier_type']
model = create_image_classifier(
    classifier_type=CONFIG['classifier_type'],
    num_classes=num_classes,
    in_channels=num_channels,
    input_size=image_size,
    learning_rate=CONFIG['learning_rate'],
    class_names=class_names,
    dataset_mean=dataset_config['mean'], # TODO: why this?
    dataset_std=dataset_config['std'],
    # CNN-specific parameters (ignored if classifier_type='mlp')
    cnn_num_conv_layers=CONFIG['cnn_num_conv_layers'],
    cnn_base_channels=CONFIG['cnn_base_channels'],
    # MLP-specific parameters (ignored if classifier_type='cnn')
    mlp_hidden_sizes=CONFIG['mlp_hidden_sizes'],
    # Common parameters
    dropout=CONFIG['dropout'],
)

print(f"Initialized {CONFIG['classifier_type'].upper()} for {dataset_config['name']}")
print(f"Input: {num_channels} x {image_size} x {image_size}")
print(f"Output: {num_classes} classes")
if CONFIG['classifier_type'] == 'cnn':
    print(f"Architecture: {CONFIG['cnn_num_conv_layers']} convolutional blocks, base_channels={CONFIG['cnn_base_channels']}")
else:
    print(f"Architecture: MLP with hidden layers {CONFIG['mlp_hidden_sizes']}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Now we'll train the model using PyTorch Lightning's Trainer with W&B logging.

In [ ]:
# Train the model (W&B logger created automatically)
trainer = create_trainer(
    max_epochs=CONFIG['max_epochs'],
    log_every_n_steps=CONFIG['log_every_n_steps'],
    wandb_project=CONFIG['wandb_project'],
    wandb_run_name=CONFIG['wandb_run_name'],
    wandb_config=CONFIG,
    model=model
)
trainer.fit(model, train_loader, test_loader)

print("✓ Training complete. View metrics and visualizations in W&B dashboard.")

Now we'll evaluate the model on the test set. 

**Note**: The trainer automatically saves checkpoints during training and loads the **best model** (based on lowest validation loss) for testing. This ensures we're evaluating the best performing model, not just the model from the last epoch.

In [ ]:
# Show best checkpoint information
if trainer.checkpoint_callback:
    print(f"Best model checkpoint: {trainer.checkpoint_callback.best_model_path}")
    print(f"Best validation loss: {trainer.checkpoint_callback.best_model_score:.4f}")
    print()

# Test the model (automatically loads best checkpoint)
test_results = trainer.test(model, test_loader, verbose=False)

# Calculate and display test accuracy
test_acc = test_results[0]['test_acc']
print(f"\n{'='*50}")
print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"{'='*50}\n")

# Log to W&B
wandb.log({'test_accuracy': test_acc * 100})

## References

- [CNN Explainer](https://poloclub.github.io/cnn-explainer/)